In [ ]:
!pip install nltk

In [ ]:
import nltk

nltk.download("all")

In [ ]:
# サンプルテキスト
text = "Natural language processing enables computers to understand human language."

# トークン化
tokens = nltk.word_tokenize(text)

# 品詞分解
pos_tags = nltk.pos_tag(tokens)

print(pos_tags)

In [4]:
import re

# サンプルテキスト
text = "自然言語処理（NLP）は、コンピュータが人間の言語を理解するための技術です。 詳しくは https://example.com をご覧ください！※この**マークダウン**記号は重要かもしれません。。。#AI #機械学習"

# HTMLタグの除去
text = re.sub(r"<.*?>", "", text)

# URLの除去
text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)

# 特殊文字の除去
text = re.sub(r"\W", " ", text)

print(text)

自然言語処理 NLP は コンピュータが人間の言語を理解するための技術です  詳しくは  をご覧ください  この  マークダウン  記号は重要かもしれません    AI  機械学習


In [8]:
# サンプルテキスト　日本語の品詞分解
# !pip install sudachidict_core

from sudachipy import Dictionary
from sudachipy import Tokenizer

tokenizer = Dictionary().create()
mode = Tokenizer.SplitMode.C  # 最も細かく分割するモード

text = "パリの魅力をたっぷり満喫できる感動ののパリ 6日間ツアー！観光名所巡りやグルメ、ショッピングを楽しみながら特別な時間を過ごしましょう。"

tokens = [(m.surface(), m.part_of_speech()) for m in tokenizer.tokenize(text, mode)]

for token, pos in tokens:
    print(token, pos)

パリ ('名詞', '固有名詞', '地名', '一般', '*', '*')
の ('助詞', '格助詞', '*', '*', '*', '*')
魅力 ('名詞', '普通名詞', '一般', '*', '*', '*')
を ('助詞', '格助詞', '*', '*', '*', '*')
たっぷり ('副詞', '*', '*', '*', '*', '*')
満喫 ('名詞', '普通名詞', 'サ変可能', '*', '*', '*')
できる ('動詞', '非自立可能', '*', '*', '上一段-カ行', '連体形-一般')
感動 ('名詞', '普通名詞', 'サ変可能', '*', '*', '*')
の ('助詞', '格助詞', '*', '*', '*', '*')
の ('助詞', '格助詞', '*', '*', '*', '*')
パリ ('名詞', '固有名詞', '地名', '一般', '*', '*')
  ('空白', '*', '*', '*', '*', '*')
6 ('名詞', '数詞', '*', '*', '*', '*')
日間 ('名詞', '普通名詞', '助数詞可能', '*', '*', '*')
ツアー ('名詞', '普通名詞', '一般', '*', '*', '*')
！ ('補助記号', '句点', '*', '*', '*', '*')
観光 ('名詞', '普通名詞', 'サ変可能', '*', '*', '*')
名所 ('名詞', '普通名詞', '一般', '*', '*', '*')
巡り ('名詞', '普通名詞', '一般', '*', '*', '*')
や ('助詞', '副助詞', '*', '*', '*', '*')
グルメ ('名詞', '普通名詞', '形状詞可能', '*', '*', '*')
、 ('補助記号', '読点', '*', '*', '*', '*')
ショッピング ('名詞', '普通名詞', 'サ変可能', '*', '*', '*')
を ('助詞', '格助詞', '*', '*', '*', '*')
楽しみ ('動詞', '一般', '*', '*', '五段-マ行', '連用形-一般')
ながら ('助詞', '接続助詞',

In [9]:
# 名詞・動詞のみ抽出
tokens = []
for m in tokenizer.tokenize(text, mode):
    pos = m.part_of_speech()[0]  # 品詞の大分類（例：名詞、動詞など）
    if pos in ["名詞", "動詞"]:
        tokens.append((m.surface(), pos))


# 出力
for token, pos in tokens:
    print(f"{token} : {pos}")

パリ : 名詞
魅力 : 名詞
満喫 : 名詞
できる : 動詞
感動 : 名詞
パリ : 名詞
6 : 名詞
日間 : 名詞
ツアー : 名詞
観光 : 名詞
名所 : 名詞
巡り : 名詞
グルメ : 名詞
ショッピング : 名詞
楽しみ : 動詞
時間 : 名詞
過ごし : 動詞


In [ ]:
# OS側ライブラリ（必要に応じて）
# !apt-get -y install mecab libmecab-dev mecab-ipadic-utf8

# Pythonバインディング
# !pip install mecab-python3 unidic-lite

zsh:1: command not found: apt-get


In [13]:
# 文書ごとに出現する単語を件数をカウントするコード
import pandas as pd
import MeCab
from collections import Counter


# サンプル文章
texts = [
    "パリにいつかは訪れてみたい　パリにはたくさんの美術館があります",
    "パリへの旅行代金はいくらになりますか？　美術館のチケットも手配可能でしょうか？",
    "ルーブル美術館のオプショナルツアー注文が完了していないようです",
    "広島からパリとロンドンを周遊する航空券を購入したい",
    "パリまで東京から往復の航空券を購入します",
]


# 分かち書き関数
def tokenize(text):
    mecab = MeCab.Tagger("-Owakati")
    return mecab.parse(text).strip().split()


# 各文章を分かち書きして頻度を数える
doc_counters = [Counter(tokenize(text)) for text in texts]


# 全単語の一覧を取得
all_words = sorted(set(word for counter in doc_counters for word in counter.keys()))


# 各文書ごとの単語頻度ベクトルを作成
rows = []
for counter in doc_counters:
    row = [counter.get(word, 0) for word in all_words]
    rows.append(row)

# DataFrame化
df_freq = pd.DataFrame(rows, columns=all_words)
df_freq.index = [f"文書{i+1}" for i in range(len(texts))]


# 表示
df_freq

,あり,い,いくら,いつ,か,から,が,し,する,たい,...,手配,旅行,東京,注文,美術,航空,訪れ,購入,館,？
文書1,1,0,0,1,1,0,1,0,0,1,...,0,0,0,0,1,0,1,0,1,0
文書2,0,0,1,0,2,0,0,0,0,0,...,1,1,0,0,1,0,0,0,1,2
文書3,0,1,0,0,0,0,1,1,0,0,...,0,0,0,1,1,0,0,0,1,0
文書4,0,0,0,0,0,1,0,1,1,1,...,0,0,0,0,0,1,0,1,0,0
文書5,0,0,0,0,0,1,0,1,0,0,...,0,0,1,0,0,1,0,1,0,0


In [ ]:
# 文書ごとに出現する単語が出現したときに1とカウントするコード
import pandas as pd
import MeCab
import os

# 環境変数の設定（必要に応じて）
# os.environ['MECABRC'] = '/etc/mecabrc'  # 適宜変更 - This is not needed if the dictionary path is specified in Tagger initialization

# サンプル文章
texts = [
    "パリにいつかは訪れてみたい　パリにはたくさんの美術館があります",
    "パリへの旅行代金はいくらになりますか？　美術館のチケットも手配可能でしょうか？",
    "ルーブル美術館のオプショナルツアー注文が完了していないようです",
    "広島からパリとロンドンを周遊する航空券を購入したい",
    "パリまで東京から往復の航空券を購入します",
]


# 分かち書き関数
def tokenize(text):
    # MeCabの初期化時に辞書ファイルのパスを指定
    mecab = MeCab.Tagger(
        "-Owakati -d /usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ipadic-neologd"
    )
    return mecab.parse(text).strip().split()


# 各文章を分かち書き
token_lists = [tokenize(text) for text in texts]

# 出現単語の集合
all_words = sorted(set(word for tokens in token_lists for word in tokens))

# ワンホットベクトル作成
binary_matrix = []
for tokens in token_lists:
    row = [1 if word in tokens else 0 for word in all_words]
    binary_matrix.append(row)

# DataFrame化（各行＝文章、各列＝単語、値＝1/0）
df_onehot = pd.DataFrame(binary_matrix, columns=all_words)

# 単語ごとの出現件数（列ごとに合計）
word_counts = df_onehot.sum().astype(int)
df_onehot.loc["出現件数"] = word_counts

# 結果表示
df_onehot

In [ ]:
# 必要なライブラリをインストール
# !pip install -q mecab-python3 unidic-lite pandas


In [ ]:
import pandas as pd
import MeCab
from collections import Counter

# CSVファイルの読み込み
df = pd.read_csv("../data/contact_history.csv")
df

In [ ]:
# MeCabの形態素解析を行う関数
def tokenize(text):
    mecab = MeCab.Tagger()  # 解析結果を全部取得
    node = mecab.parseToNode(text)
    nouns = []
    while node:
        if node.feature.split(",")[0] == "名詞":  # 名詞だけ抽出
            nouns.append(node.surface)
        node = node.next
    return nouns


# 問い合わせ内容 (inquiry) を分かち書きしてリスト化
df["tokens"] = df["inquiry"].astype(str).apply(tokenize)

# すべての単語をカウント
all_tokens = [word for tokens in df["tokens"] for word in tokens]
word_counts = Counter(all_tokens)

# 上位のキーワードを取得
top_keywords = word_counts.most_common(10)

# 結果をデータフレーム化
output_df = pd.DataFrame(top_keywords, columns=["キーワード", "出現回数"])

# 結果を表示
print(output_df)